# 347. Top K Frequent Elements
**Difficulty:** 🟡 Medium · **Topic:** Heap · **LeetCode:** https://leetcode.com/problems/top-k-frequent-elements/

## 💡 Concepts

**Core concept(s):** Count frequencies, then pick the top k with a **heap** — or with **bucket sort** in linear time.

**Why it applies here:** First count how often each value appears. To get the k most frequent, a heap keeps the top k in O(n log k); even faster, bucket the values by frequency (frequency can't exceed n) and read buckets from the top.

**Key intuition:** Count first; then either keep the k biggest with a heap, or bucket values by their count.

---

### 📚 What is a Heap (Priority Queue)?
A **heap** always gives its smallest (min-heap) or largest (max-heap) item in **O(log n)**. Ideal for "keep the top K" or "always grab the current extreme".
- **In Python:** `heapq` (a min-heap; negate values for a max-heap).

---

**Prerequisite knowledge:**
- A frequency `dict`/`Counter`.
- A heap, or bucketing by frequency.

## 📝 Problem

Return the `k` most frequent elements.

**Example**
```
nums=[1,1,1,2,2,3], k=2 -> [1,2]
```

> Two approaches: heap `O(n log k)` and bucket sort `O(n)`.

### Approach 1 — Count + Heap

**Idea:** Count frequencies, then use a heap to grab the k largest counts.

**Time:** `O(n log k)`. **Space:** `O(n)`.

In [ ]:
import heapq
from collections import Counter

def top_k_heap(nums, k):
    counts = Counter(nums)                 # value -> how many times it appears
    # nlargest keeps only the k items with the biggest counts (a size-k heap under the hood).
    return [val for val, _ in heapq.nlargest(k, counts.items(), key=lambda x: x[1])]

### Approach 2 — Bucket Sort (optimal)

**Idea:** A value's frequency is between 1 and n. Put each value in the bucket for its frequency, then read buckets from highest frequency down until you have k.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
from collections import Counter

def top_k_bucket(nums, k):
    counts = Counter(nums)                 # value -> frequency
    buckets = [[] for _ in range(len(nums) + 1)]   # index = frequency (max possible is n)
    for val, freq in counts.items():
        buckets[freq].append(val)          # drop each value into its frequency bucket
    res = []
    for freq in range(len(buckets) - 1, 0, -1):    # read buckets from highest frequency down
        for val in buckets[freq]:
            res.append(val)
            if len(res) == k:              # collected the k most frequent
                return res
    return res

In [ ]:
# Correctness check (order among equal-frequency items may vary)
tests = [([1,1,1,2,2,3],2,{1,2}), ([1],1,{1}), ([4,4,4,6,6,2],2,{4,6})]
for nums, k, exp in tests:
    a, b = set(top_k_heap(nums, k)), set(top_k_bucket(nums, k))
    print(f"{nums}, k={k} -> heap={a}, bucket={b}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n²)`     | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    nums = [i % (n // 4 + 1) for i in range(n)]
    return (nums, 10)
solutions = {
    "heap   O(n log k)": top_k_heap,
    "bucket O(n)      ": top_k_bucket,
}
sizes = [20000, 40000, 80000, 160000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Count then select:** frequency map first; then a heap for top-k, or buckets for linear time.
- **Bucket by a bounded key:** when frequencies are bounded by n, bucketing beats sorting.
- **Signal:** "top k / k most frequent / k largest".
- **Related problems:** K Closest Points, Kth Largest Element, Sort Characters by Frequency.
- **Common pitfalls:** (1) sorting everything (O(n log n)) when top-k suffices; (2) empty-bucket handling.